# GPT-4o NER coursework archive

Original coursework: June 2025, Natural Language Processing, University of Padova, Prof. Giorgio Satta. Published on GitHub: November 2026.

The submission listed Farzam Nikbakhsh Jorshari and Amirreza Soltani. Farzam was the primary developer and carried out the implementation and experiments. Both names remain credited.

This November 2026 maintenance copy removes credentials, saved outputs and student IDs, fixes CoNLL-2003 labels/BIO alignment, and aborts ambiguous alignment or failed API responses. The three prompt structures remain; the demonstration condition is few-shot. Original prose describes the historical coursework, not a new evaluation.

Historical scores used faulty label conversion and are unverified; see docs/historical-evaluation.md. No API experiment was rerun for this release. The original gpt-4o alias has no recorded snapshot, so later calls cannot establish identical 2025 behavior.

The notebook requires OPENAI_API_KEY and CONLL2003_PATH in the environment. The latter must point to a lawfully obtained, local Hugging Face DatasetDict export with validation and test splits, tokens, and ClassLabel-based ner_tags. No data or saved model responses are distributed. Running the experiment cells makes paid API requests. Install requirements.txt before opening the notebook.


# Part 2 of NLP Project 2025: Zero-Shot NER with GPT-4o

**Names:** Farzam Nikbakhsh Jorshari & Amirreza Soltani  
**Degree Program:** Computer Engineering  
**Working Environment:** Jupyter Notebook

This notebook presents our implementation of zero-shot Named Entity Recognition (NER) using GPT-4o, following the guidelines provided in the project description and inspired by the paper "Empirical Study of Zero-Shot NER with ChatGPT" by Xie et al. (2023).  
We chose to experiment with the instruction-tuned GPT-4o API, taking advantage of its robust performance.

**In the notebook we have provided a description for each cell of the code, in a separate Markdown-cell, below it.**


### References

- Xie, Tingyu, et al. (2023). *Empirical Study of Zero-Shot NER with ChatGPT*.  
  [https://arxiv.org/abs/2310.10035](https://arxiv.org/abs/2310.10035)

- Munnangi, Monica (2024). *A Brief History of Named Entity Recognition*.  
  [https://arxiv.org/abs/2411.05057](https://arxiv.org/abs/2411.05057)


## Dataset Description

For this project, we worked with the **CoNLL-2003** dataset which is a widely used benchmark dataset for Named Entity Recognition in the general newswire domain. The dataset is annotated in BIO format and includes the following entity types:

- **PER** : Person  
- **ORG** : Organization  
- **LOC** : Location  
- **MISC** : Miscellaneous  

Each sample consists of a sentence broken into tokens, where each token is labeled with its corresponding entity type or marked as outside of any entity.  
Some of the common challenges with this dataset include:

- Multi-token entity names  
- Ambiguous boundaries between entity types  
- Nested structures not explicitly annotated  

We decided to go with CoNLL-2003 because it’s standard, well-annotated, and offers a fair challenge for testing zero-shot prompts.


In [ ]:
# Install requirements.txt in your environment before opening this notebook.


In [ ]:
import openai
import time
import json
from datasets import DatasetDict, Dataset
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import os
import logging
import re

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

Basic imports for working with data, API querying, evaluation, and logging. Nothing too fancy here.

## API Setup

We used the OpenAI API with the **GPT-4o** model.  
While the professor suggested trying free models like Gemini, we had already purchased GPT-4o access for another project, so we used it here too.

In [ ]:
openai.api_key = os.environ["OPENAI_API_KEY"]


This sets up access to GPT-4o through OpenAI's API.

## Dataset
Development: 50 validation sentences. Historical test: 50 test sentences. Both shuffled with seed 42.


In [ ]:
from datasets import load_from_disk

dataset_path = os.environ.get("CONLL2003_PATH")
if not dataset_path:
    raise RuntimeError("Set CONLL2003_PATH to an authorized local DatasetDict export.")
conll = load_from_disk(dataset_path)
expected_tags = {"O"} | {prefix + label for prefix in ("B-", "I-")
                        for label in ("PER", "ORG", "LOC", "MISC")}
for split in ("validation", "test"):
    if split not in conll or len(conll[split]) < 50:
        raise ValueError("Both validation and test splits need at least 50 sentences.")
    names = conll[split].features["ner_tags"].feature.names
    if set(names) != expected_tags:
        raise ValueError("Unexpected CoNLL-2003 BIO label schema.")
if conll["validation"].features["ner_tags"] != conll["test"].features["ner_tags"]:
    raise ValueError("Validation and test numeric-label schemas differ.")

sample_size = 50
dev_samples = conll["validation"].shuffle(seed=42).select(range(sample_size))

id2tag = dict(enumerate(conll['validation'].features['ner_tags'].feature.names))
tag2id = {tag: idx for idx, tag in id2tag.items()}

texts = [" ".join(tokens) for tokens in dev_samples["tokens"]]
true_data = [{"tokens": tokens, "ner_tags": tags} 
             for tokens, tags in zip(dev_samples["tokens"], dev_samples["ner_tags"])]


The integer-to-tag mapping comes from the local dataset features, rather than an assumed numeric order. Validation and test must use the same nine BIO labels. Development samples exactly 50 validation sentences with shuffle seed 42.


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

#Count entity tags in the dev set
entity_counts = Counter(tag for sample in true_data for tag in sample["ner_tags"])

#Map integer tags back to string labels
entity_labels = [id2tag[i] for i in entity_counts.keys()]
counts = list(entity_counts.values())

plt.figure(figsize=(8, 4))
plt.bar(entity_labels, counts, color='skyblue')
plt.title("Distribution of Entity Types in Sample")
plt.xlabel("Entity Type")
plt.ylabel("Token Count")
plt.tight_layout()
plt.show()


We plotted the distribution of entity types in our 50-sample subset of the CoNLL-2003 validation set. This gives us a quick sense of which classes are most represented in the evaluation data; especially useful for interpreting model scores later.

### Prompt Templates for Each Method

Before implementing the prediction methods, we briefly summarize the structure of the prompts we used. Each one is adapted from Xie et al. (2023) and tailored for our GPT-4o setup.

| **Method**      | **Prompt Type**       | **Prompt Style Example**                                                                 |
|-----------------|------------------------|-------------------------------------------------------------------------------------------|
| **Baseline**    | Direct extraction      | *You are an NER system. Extract all named entities from the text and classify them as PERSON, ORGANIZATION, etc. Return in JSON.* |
| **Explanation** | Justified reasoning    | *Extract named entities from the text and explain why each is labeled as such. Return JSON with explanation.* |
| **Demonstration** | Few-shot examples    | *Here are a few examples: ‘Barack Obama’ → PERSON, ‘Google’ → ORGANIZATION... Now extract from this text.* |

We kept the prompts structured and clear to help the model generate outputs that are easy to interpret and evaluate.


In [ ]:
class OptimizedZeroShotNER:
    def __init__(self, model_name="gpt-4o", rate_limit=60):
        self.model_name = model_name
        self.rate_limit = rate_limit
        self.last_request_time = 0
        self.request_count = 0
        self.entity_types = ["PERSON", "ORGANIZATION", "LOCATION", "MISC"]
        logging.info("OpenAI GPT model initialized.")

    def enforce_rate_limit(self):
        elapsed = time.time() - self.last_request_time
        if elapsed < 60 and self.request_count >= self.rate_limit:
            sleep_time = 60 - elapsed + 0.1
            logging.info(f"Rate limit reached. Sleeping for {sleep_time:.2f} seconds...")
            time.sleep(sleep_time)
            self.request_count = 0
        elif elapsed >= 60:
            self.request_count = 0
        self.last_request_time = time.time()
        self.request_count += 1

    def create_prompt(self, text, method="baseline"):
        if method == "baseline":
            return f"""You are an NER system. Extract all named entities from the following text. 
Classify each as one of: {", ".join(self.entity_types)}.

Return the result as JSON in this format:
{{
  "entities": [
    {{"entity_text": "John Smith", "entity_type": "PERSON"}},
    {{"entity_text": "Microsoft", "entity_type": "ORGANIZATION"}}
  ]
}}

Text: {text}"""

        elif method == "explanation":
            return f"""Extract all named entities from the following text and explain why each is labeled as such.
Entity types: {", ".join(self.entity_types)}.
Return JSON: 'entity_text', 'entity_type', 'explanation'.
Text: {text}"""

        elif method == "demonstration":
            return f"""Here are examples of named entity classifications:
- "John Smith" → PERSON
- "Microsoft" → ORGANIZATION
- "Paris" → LOCATION

Now extract entities from this text:
Text: {text}
Return JSON: 'entity_text' and 'entity_type' for each entity."""

        else:
            raise ValueError(f"Unknown method: {method}")

    def query_llm(self, prompt_text):
        self.enforce_rate_limit()
        try:
            response = openai.chat.completions.create(
                model=self.model_name,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant for Named Entity Recognition (NER)."},
                    {"role": "user", "content": prompt_text}
                ],
                temperature=0.3
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            raise RuntimeError("API request failed; abort evaluation instead of scoring empty predictions.") from None

    def parse_response(self, response):
        try:
            json_match = re.search(r'\{.*\}|\[.*\]', response, re.DOTALL)
            if not json_match:
                raise ValueError("No JSON entity response")

            parsed = json.loads(json_match.group())

            if isinstance(parsed, dict) and "entities" in parsed:
                entities = parsed["entities"]
            elif isinstance(parsed, list):
                entities = parsed
            else:
                raise ValueError("Expected an entities object or list")
            if not isinstance(entities, list) or any(
                not isinstance(entity, dict)
                or not isinstance(entity.get("entity_text"), str)
                or not entity["entity_text"].strip()
                or not isinstance(entity.get("entity_type"), str)
                for entity in entities
            ):
                raise ValueError("Malformed entity schema")
            return entities
        except (ValueError, TypeError):
            raise ValueError("Invalid JSON entity response; abort evaluation.") from None

    def predict(self, texts, method="baseline", batch_size=5):
        results = []
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Predicting ({method})"):
            batch = texts[i:i + batch_size]
            for text in batch:
                prompt = self.create_prompt(text, method)
                response = self.query_llm(prompt)
                entities = self.parse_response(response)
                results.append({"text": text, "entities": entities})
        return results


We put everything inside this class to keep things tidy and reusable. It handles the prompts, calls the API, checks for rate limits (so we don’t go over the quota), and extracts the final entities. We also support three different methods, one that just returns the entities, one that also explains them, and one that works by showing examples. We tried to follow the structure used in the Xie et al. paper, while adapting it to our own format.

### Evaluation protocol

Use exactly 50 validation sentences to develop prompts. Freeze prompts and alignment policy before running the 50-sentence test subset. Strict BIO alignment aborts ambiguous, overlapping or unmatched entity spans; review these cases rather than silently dropping them from metrics. seqeval uses strict IOB2 entity matching. This release has no corrected model scores.


## Baseline Method

This is the most straightforward method we tried. It follows the basic setup from the paper, where the model is simply asked to find named entities in the text and label them using a list of possible types (like PERSON, LOCATION, etc.).

The idea is to see how well the model can handle the task with just a plain instruction and no examples or extra explanation. The output is expected in a clean JSON format where each entity has a type. It’s basically testing how far we can get using just the model’s general knowledge and instruction-following ability.


In [ ]:
ner = OptimizedZeroShotNER(model_name="gpt-4o")
logging.info("Model initialized and ready.")

baseline_predictions = ner.predict(texts, method="baseline")
for i, result in enumerate(baseline_predictions[:3]):
    print(f"\nSentence {i+1}")
    print("Text:", result["text"])
    print("Entities:", result["entities"])

We initialized our model and used it to generate predictions with the baseline prompt structure. To check if everything works as expected, we printed the output for three sample inputs; this helps verifying that the model is correctly parsing and returning entities in JSON.

In [ ]:
from seqeval.metrics import classification_report
from bio_alignment import align_predictions_to_bio

y_true, y_pred = align_predictions_to_bio(true_data, baseline_predictions, id2tag)
print(classification_report(y_true, y_pred, digits=4, mode="strict", scheme=IOB2, zero_division=0))


We aligned the predicted entities from the model to the BIO tagging format used in the dataset. This was necessary since the model outputs span-level entities, but the evaluation requires token-level labels. After alignment, we computed precision, recall, and F1-score using seqeval to assess how the baseline method performed.

Historical results are kept separately in docs/historical-evaluation.md and are not corrected performance claims. This output-cleared notebook has not been rerun against the API.


## Explanation Method

This method is also taken from Xie et al. (2023), but it's a bit more detailed than the baseline. Instead of just asking the model to find and label entities, we also ask it to explain why each prediction is assigned to a certain type.

The idea behind this is that if the model has to explain its reasoning, it might think more carefully about the labels it assigns. Of course, for evaluation, we just extract the labels and ignore the explanations, but they could help with overall accuracy. This prompt is more reflective and might guide the model to make more consistent predictions.


In [ ]:
explanation_predictions = ner.predict(texts, method="explanation")
y_true_exp, y_pred_exp = align_predictions_to_bio(true_data, explanation_predictions, id2tag)

print("\n Evaluation Report for Explanation Method:\n")
print(classification_report(y_true_exp, y_pred_exp, digits=4, mode="strict", scheme=IOB2, zero_division=0))


We evaluated the explanation-based prompting strategy. While this method asks the model to justify its entity classifications, we focused only on the extracted entity types for evaluation. As before, we aligned the outputs to the token-level BIO format and reported precision, recall, and F1-score to measure its performance.

## Demonstration Method

This method is based on in-context learning from Xie et al. (2023). We give the model a few short examples (like “John Smith → PERSON”) before the actual text. The idea is to "show" the model how the task works with minimal examples, kind of like how people learn from seeing patterns.

It doesn’t include explanations - just examples- so it's lighter than the Explanation method. This type of prompting is meant to help the model generalize by mimicking the format of the output directly after seeing a couple of samples.


In [ ]:
demo_predictions = ner.predict(texts, method="demonstration")

y_true_demo, y_pred_demo = align_predictions_to_bio(true_data, demo_predictions, id2tag)

print("\nEvaluation Report for Demonstration Method:\n")
print(classification_report(y_true_demo, y_pred_demo, digits=4, mode="strict", scheme=IOB2, zero_division=0))


We applied the demonstration method to predict entities from the same input texts. Once we got the predictions, we aligned them with the BIO tagging format of the dataset to make the evaluation work properly. Then we calculated the precision, recall, and F1-score to see how this method performed overall.


In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

#Flatten predictions
baseline_tags = [tag for seq in y_pred for tag in seq]
explanation_tags = [tag for seq in y_pred_exp for tag in seq]
demonstration_tags = [tag for seq in y_pred_demo for tag in seq]

#Count occurrences
baseline_counts = Counter(baseline_tags)
explanation_counts = Counter(explanation_tags)
demo_counts = Counter(demonstration_tags)

#Ensure consistent tag order
all_tags = sorted(set(baseline_counts) | set(explanation_counts) | set(demo_counts))
baseline_vals = [baseline_counts.get(tag, 0) for tag in all_tags]
explanation_vals = [explanation_counts.get(tag, 0) for tag in all_tags]
demo_vals = [demo_counts.get(tag, 0) for tag in all_tags]

x = range(len(all_tags))
bar_width = 0.25

plt.figure(figsize=(12, 6))
plt.bar([i - bar_width for i in x], baseline_vals, width=bar_width, label='Baseline', alpha=0.8, color='#FDBE85')
plt.bar(x, explanation_vals, width=bar_width, label='Explanation', alpha=0.8, color='#FC9272')
plt.bar([i + bar_width for i in x], demo_vals, width=bar_width, label='Demonstration', alpha=0.8, color='#DE2D26')
plt.xticks(x, all_tags, rotation=45)
plt.xlabel("BIO Tag")
plt.ylabel("Count")
plt.title("Predicted Tag Distribution Comparison Across Methods")
plt.legend()
plt.tight_layout()
plt.show()


We compared how each method distributed its predicted tags across the BIO entity types. This chart helps us visualize patterns, like over-prediction of the “O” tag or under-detection of certain entity types. It also provides a clear view of how each method prioritizes different classes and can guide error analysis or prompt refinement decisions.

### Final test subset

The following cell evaluates each fixed prompt on 50 shuffled test sentences (seed 42). It makes API requests when executed. The release contains no outputs from this maintenance version; record any later evaluation as a new experiment, including dataset revision and response model identity.


In [ ]:
test_samples = conll["test"].shuffle(seed=42).select(range(50))

texts_test   = [" ".join(toks) for toks in test_samples["tokens"]]
true_test    = [
    {"tokens": toks, "ner_tags": tags}
    for toks, tags in zip(test_samples["tokens"], test_samples["ner_tags"])
]

methods = ["baseline", "explanation", "demonstration"]
final_reports = {}
for m in methods:
    preds = ner.predict(texts_test, method=m)
    y_t, y_p = align_predictions_to_bio(true_test, preds, id2tag)
    rep = classification_report(y_t, y_p, digits=4, output_dict=True, mode="strict", scheme=IOB2, zero_division=0)
    final_reports[m] = rep

import pandas as pd
df_final = pd.DataFrame({
    m: [final_reports[m]["weighted avg"]["precision"],
        final_reports[m]["weighted avg"]["recall"],
        final_reports[m]["weighted avg"]["f1-score"]]
    for m in methods
}, index=["Precision", "Recall", "F1 (wtd)"]).T

df_final


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ["Precision", "Recall", "F1 (wtd)"]
colors = ["#FFA07A", "#FF6347", "#DC143C"]

x = np.arange(len(df_final.index))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))

for i, metric in enumerate(metrics):
    values = df_final[metric].values
    ax.bar(x + i * width - width, values, width, label=metric, color=colors[i])

ax.set_ylabel('Score')
ax.set_title('Final Test Set Evaluation (All Methods)')
ax.set_xticks(x)
ax.set_xticklabels(df_final.index)
ax.legend()

for i, metric in enumerate(metrics):
    for j, val in enumerate(df_final[metric].values):
        ax.annotate(f"{val:.2f}", (j + i * width - width, val + 0.005), ha='center', fontsize=8)

plt.tight_layout()
plt.show()
